# Week 10 — Available-reader run: Mistral + Qwen; Llama pending

User explicitly authorized running the available study now and deferring gated Llama access. Run All resumes completed R/C, qualifies Mistral/Qwen, runs their G surfaces, executes the prespecified Mistral D diagnostic with a documented partial-panel gate exception, and refreshes reports. Llama stays in the frozen panel and is marked pending; full G is never falsely marked complete. No policy learning or QASPER TEST use.


In [1]:
from pathlib import Path
import sys, json, subprocess
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
while not (ROOT/'src/week10/root_cause').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Open within CS787-RAG-Project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.week10.root_cause.protocol import load
CONFIG = ROOT/'configs/week10/root_cause_v1.json'
cfg = load(CONFIG)
PYTHON = ROOT/'sara_env/bin/python'
RESULT_ROOT = ROOT/cfg['output_root']
ACTIVE_READERS = ('Mistral', 'Qwen')  # Llama explicitly deferred by the user.
SMOKE_ONLY = False  # True = safe preview; False = resume the approved study with Run All.
# Stage setup cells below assign these automatically; do not edit them here.
RUN_AUDIT = True
RUN_TINY_MODEL_SMOKE = False
RUN_RETRIEVER_FULL = False
RUN_COMPRESSION_FULL = False
RUN_READER_FULL = False
RUN_DATASET_ANALYSIS_FULL = False
RUN_POLICY_EXPERIMENT = False
RUN_REPORT = False

def stage(phase, *args, execute=False):
    command = [str(PYTHON), '-m', 'src.week10.root_cause.runner', '--config', str(CONFIG), '--phase', phase, *args]
    if execute: command.append('--execute')
    subprocess.run(command, cwd=ROOT, check=True)


In [2]:
from src.week10.root_cause.storage import Store, digest, file_hash, atomic
from src.week10.root_cause.protocol import source_hashes

# This notebook changes scheduling only. Frozen method code/config is untouched.
def workflow_identity():
    path = RESULT_ROOT/'execution_freeze.json'
    if not path.exists():
        return None
    saved = json.loads(path.read_text())
    if saved['config'] != cfg or saved['sources'] != source_hashes():
        raise ValueError('Frozen config/source changed. Resume with the original configuration.')
    return digest(saved)

FROZEN_RUN_ID = workflow_identity()
QUERY_PAPERS = {}
with Path(cfg['dataset']).open() as stream:
    for line in stream:
        if line.strip():
            row = json.loads(line)
            QUERY_PAPERS[str(row['id'])] = str(row['example_id'])
if cfg.get('query_limit'):
    QUERY_PAPERS = dict(list(QUERY_PAPERS.items())[:cfg['query_limit']])
QUERY_IDS = list(QUERY_PAPERS)

def expected_keys(kind):
    if kind == 'retrieval': return QUERY_IDS
    if kind == 'R': return [[q, k] for q in QUERY_IDS for k in cfg['k_values']]
    if kind == 'C':
        return [[q, arm] for q in QUERY_IDS for arm in ('no_evidence', 'full', 'sara_k4', 'lead256', 'qcs256')]
    if kind == 'G':
        return [[q, arm] for q in QUERY_IDS for arm in ('no_evidence', 'full', 'qcs256')]
    if kind == 'D':
        subset = sorted(QUERY_IDS, key=lambda q: digest([QUERY_PAPERS[q], q, 42]))[:24]
        return [[q, seed, arm] for q in subset for seed in (42, 43, 44) for arm in ('no_evidence', 'full')]
    raise ValueError('Unknown checkpoint surface: '+kind)

def checkpoint_progress(root, name, keys):
    directory = root/'runs'/name
    identity_path = directory/'identity.json'
    if not identity_path.exists(): return 0, False
    identity = json.loads(identity_path.read_text())
    if root == RESULT_ROOT and FROZEN_RUN_ID is not None and identity['freeze_hash'] != FROZEN_RUN_ID:
        raise ValueError('Checkpoint belongs to a different frozen run: '+name)
    store = Store(directory, identity)
    # Validate checksums and expected query/action keys, not just filenames/counts.
    saved = sum(store.get(key) is not None for key in keys)
    complete = saved == len(keys) and (directory/'complete.json').exists()
    return saved, complete

def run_pending(name, kind, phase, *args):
    keys = expected_keys(kind)
    saved, complete = checkpoint_progress(RESULT_ROOT, name, keys)
    if complete:
        print(f'SKIP {name}: {saved}/{len(keys)} validated; no model load or generation.')
        return
    print(f'RESUME {name}: {saved}/{len(keys)} saved; only pending work will execute.', flush=True)
    stage(phase, *args, execute=True)

def phase_approved(phase):
    path = RESULT_ROOT/f'freeze_{phase}.json'
    if not path.exists(): return False
    obj = json.loads(path.read_text())
    if obj.get('phase') != phase or obj.get('GO_NEXT') is not True:
        raise ValueError('Invalid phase review: '+phase)
    current = workflow_identity()
    if current is None or obj.get('execution_freeze_hash') != current:
        raise ValueError('Phase review belongs to another frozen run: '+phase)
    return True

def review_if_requested(phase):
    if phase_approved(phase):
        print(f'Reusing the existing Phase {phase} review; it will not be overwritten.')
        return
    if REVIEW_PHASE == phase:
        if REVIEWED_RETRIEVER not in ('BM25', 'MiniLM', 'SFR') or not REVIEW_REASON.strip():
            raise ValueError('Provide the reviewed retriever and evidence before approving this phase.')
        stage('RCA', '--review-phase', phase, '--retriever', REVIEWED_RETRIEVER,
              '--review-reason', REVIEW_REASON, execute=True)
    else:
        print(f'Phase {phase} review pending. Later stages will stay paused; reports/status still run.')

def refresh_reports():
    completed = []
    for identity_path in sorted((RESULT_ROOT/'runs').glob('*/identity.json')):
        directory = identity_path.parent
        if (directory/'complete.json').exists():
            completed.append([directory.name, file_hash(identity_path), file_hash(directory/'complete.json')])
    if not any(name.startswith(('R-', 'C-', 'G-')) for name, *_ in completed):
        print('No completed answer surface to report yet.')
        return
    state_path = RESULT_ROOT/'notebook_orchestration/report_state.json'
    stamp = digest([completed, workflow_identity()])
    required = ('summary.csv', 'bootstrap.csv', 'oracle_diagnostics.json', 'retrieval_summary.csv', 'decision_R.json')
    if state_path.exists():
        state = json.loads(state_path.read_text())
        recorded = state.get('report_hashes', {})
        if state.get('stamp') == stamp and all((RESULT_ROOT/n).exists() and recorded.get(n) == file_hash(RESULT_ROOT/n) for n in required):
            print('SKIP report computation: existing reports match the completed surfaces.')
            return
    stage('RCA', execute=True)
    atomic(state_path, dict(stamp=stamp, report_hashes={n:file_hash(RESULT_ROOT/n) for n in required}))


def prepare_reader_panel():
    if SMOKE_ONLY:
        print('Preview only: reader downloads and GPU qualification disabled.')
        return False
    if not phase_approved('R') or not checkpoint_progress(RESULT_ROOT, 'C-Mistral', expected_keys('C'))[1]:
        print('Reader setup paused until C completes under the reviewed R configuration.')
        return False
    if all(checkpoint_progress(RESULT_ROOT, 'G-'+reader, expected_keys('G'))[1] for reader in ACTIVE_READERS):
        print('SKIP reader preparation: every G surface is already complete.')
        return True
    print('Preparing the pinned reader panel: disk check, resumable downloads, then one-reader-at-a-time GPU qualification.', flush=True)
    subprocess.run([str(PYTHON), str(ROOT/'scripts/week10/prepare_readers.py'), '--config', str(CONFIG),
                    '--download', '--qualify', '--readers', *ACTIVE_READERS], cwd=ROOT, check=True)
    return True


def partial_stage(name):
    subprocess.run([str(PYTHON), str(ROOT/'scripts/week10/available_panel.py'),
                    '--config', str(CONFIG), '--stage', name], cwd=ROOT, check=True)

def run_partial_d():
    keys=expected_keys('D')
    saved,complete=checkpoint_progress(RESULT_ROOT,'D-stability',keys)
    if complete:
        print(f'SKIP D: {saved}/{len(keys)} validated; no model load.')
    else:
        print(f'RESUME D under explicit partial-panel authorization: {saved}/{len(keys)} saved.',flush=True)
        partial_stage('D')


## Audit — setup, then execution

In [3]:
RUN_AUDIT = True


In [4]:
if RUN_AUDIT:
    stage('audit')
if SMOKE_ONLY:
    stage('smoke')
stage('status')


{
  "audit_valid": true,
  "split": "dev",
  "full_experiments_started": false,
  "GO_NEXT": true,
  "go_scope": "synthetic checks and manually qualified Phase R only; no completed root-cause findings",
  "remaining_gates": [
    "representative GPU memory smoke",
    "faithful SeleCom adapter",
    "strong learned textual compressor",
    "reader panel feasibility/freeze",
    "fresh external confirmation exposure audit"
  ]
}


{
  "C-Mistral": {
    "saved": 1155,
    "complete": true
  },
  "R-BM25-full": {
    "saved": 924,
    "complete": true
  },
  "R-BM25-sara": {
    "saved": 924,
    "complete": true
  },
  "R-MiniLM-full": {
    "saved": 924,
    "complete": true
  },
  "R-MiniLM-sara": {
    "saved": 924,
    "complete": true
  },
  "R-SFR-full": {
    "saved": 924,
    "complete": true
  },
  "R-SFR-sara": {
    "saved": 924,
    "complete": true
  },
  "R0-BM25": {
    "saved": 231,
    "complete": true
  },
  "R0-MiniLM": {
    "saved": 231,
    "complete": true
  },
  "R0-SFR": {
    "saved": 231,
    "complete": true
  }
}


## One-query GPU qualification — automatic resume
Already completed qualification is skipped. Missing qualification runs in its separate smoke directory before the main study. No completed smoke answer is regenerated.


In [5]:
RUN_TINY_MODEL_SMOKE = not SMOKE_ONLY


In [6]:
if RUN_TINY_MODEL_SMOKE:
    import copy
    tiny = copy.deepcopy(cfg)
    tiny['query_limit'] = 1
    tiny['k_values'] = [2, 8]
    tiny['output_root'] = 'results/week10/root_cause/gpu_smoke_v1'
    tiny_config = ROOT/'configs/week10/root_cause_gpu_smoke_v1.json'
    if tiny_config.exists():
        if json.loads(tiny_config.read_text()) != tiny:
            raise ValueError('Smoke configuration changed; do not replace its saved results.')
    else: atomic(tiny_config, tiny)
    tiny_root = ROOT/tiny['output_root']
    query = QUERY_IDS[0]
    def tiny_pending(name, keys, phase, *args):
        saved, complete = checkpoint_progress(tiny_root, name, keys)
        if complete:
            print(f'SKIP GPU smoke {name}: complete; no model load.')
            return
        print(f'RESUME GPU smoke {name}: {saved}/{len(keys)} saved.', flush=True)
        subprocess.run([str(PYTHON), '-m', 'src.week10.root_cause.runner', '--config', str(tiny_config),
                        '--phase', phase, *args, '--execute'], cwd=ROOT, check=True)
    for retriever in ('BM25', 'MiniLM', 'SFR'):
        tiny_pending('R0-'+retriever, [query], 'R0', '--retriever', retriever)
    for retriever in ('BM25', 'MiniLM', 'SFR'):
        for interface in ('full', 'sara'):
            tiny_pending(f'R-{retriever}-{interface}', [[query, k] for k in (2, 8)],
                         'R', '--retriever', retriever, '--interface', interface)
else:
    print('Preview only: GPU qualification disabled.')


SKIP GPU smoke R0-BM25: complete; no model load.
SKIP GPU smoke R0-MiniLM: complete; no model load.
SKIP GPU smoke R0-SFR: complete; no model load.
SKIP GPU smoke R-BM25-full: complete; no model load.
SKIP GPU smoke R-BM25-sara: complete; no model load.
SKIP GPU smoke R-MiniLM-full: complete; no model load.
SKIP GPU smoke R-MiniLM-sara: complete; no model load.
SKIP GPU smoke R-SFR-full: complete; no model load.
SKIP GPU smoke R-SFR-sara: complete; no model load.


## W10-R — setup, then resume the three matched retrievers

In [7]:
RUN_RETRIEVER_FULL = not SMOKE_ONLY


In [8]:
if RUN_RETRIEVER_FULL:
    for retriever in ('BM25', 'MiniLM', 'SFR'):
        run_pending('R0-'+retriever, 'retrieval', 'R0', '--retriever', retriever)
    for retriever in ('BM25', 'MiniLM', 'SFR'):
        for interface in ('full', 'sara'):
            run_pending(f'R-{retriever}-{interface}', 'R', 'R', '--retriever', retriever, '--interface', interface)
else: print('Preview only: Phase R generation disabled.')


SKIP R0-BM25: 231/231 validated; no model load or generation.


SKIP R0-MiniLM: 231/231 validated; no model load or generation.


SKIP R0-SFR: 231/231 validated; no model load or generation.
SKIP R-BM25-full: 924/924 validated; no model load or generation.


SKIP R-BM25-sara: 924/924 validated; no model load or generation.
SKIP R-MiniLM-full: 924/924 validated; no model load or generation.


SKIP R-MiniLM-sara: 924/924 validated; no model load or generation.
SKIP R-SFR-full: 924/924 validated; no model load or generation.


SKIP R-SFR-sara: 924/924 validated; no model load or generation.


## Phase R reports — reuse current reports or refresh after new completions

In [9]:
RUN_REPORT = not SMOKE_ONLY


In [10]:
if RUN_REPORT: refresh_reports()


SKIP report computation: existing reports match the completed surfaces.


## Review R before C
Existing `freeze_R.json` is reused, including its recorded retriever/reason. No edits are needed for a previously reviewed phase. If no review exists, set the three fields here once after inspecting R; all later flags are automatic. Use Python `None`, never the string `'None'`.


In [11]:
REVIEW_PHASE = None  # Set to 'R' only when recording a new reviewed decision.
REVIEWED_RETRIEVER = 'SFR'
REVIEW_REASON = ''
if not SMOKE_ONLY: review_if_requested('R')


Reusing the existing Phase R review; it will not be overwritten.


## W10-C — setup, then resume representation controls
The frozen R retriever and method settings remain unchanged. QCS256 is a textual surrogate, not SeleCom. A missing R review leaves this stage paused.


In [12]:
RUN_COMPRESSION_FULL = not SMOKE_ONLY and phase_approved('R')


In [13]:
if RUN_COMPRESSION_FULL:
    run_pending('C-Mistral', 'C', 'C', '--reader', 'Mistral')
else: print('PAUSED C: preview mode or reviewed Phase R freeze required.')


SKIP C-Mistral: 1155/1155 validated; no model load or generation.


## Phase C reports — automatically refresh after completion

In [14]:
RUN_REPORT = not SMOKE_ONLY


In [15]:
if RUN_REPORT: refresh_reports()


SKIP report computation: existing reports match the completed surfaces.


## Prepare the available readers — Llama explicitly deferred

This user-authorized partial run uses the already frozen Mistral and Qwen definitions. Llama remains `PENDING_ACCESS` and will have no invented predictions or scores. No method/model revision changes and no R/C reruns are needed. Model download and qualification resumes independently. Account access for Llama can be restored later without regenerating these reader results.


In [16]:
if not SMOKE_ONLY: partial_stage('status')
READERS_READY = prepare_reader_panel()


{
  "execution_freeze_hash": "6d603f7220cf6545e64a32227fad18497ea7d56dec8013925d08ab503fbd0488",
  "full_panel_complete": false,
  "available_panel_complete": false,
  "active_readers": [
    "Mistral",
    "Qwen"
  ],
  "coverage": {
    "Mistral": {
      "saved": 0,
      "expected": 693,
      "complete": false
    },
    "Qwen": {
      "saved": 0,
      "expected": 693,
      "complete": false
    }
  },
  "pending_readers": {
    "Llama": {
      "status": "PENDING_ACCESS",
      "model": {
        "id": "meta-llama/Llama-3.1-8B-Instruct",
        "revision": "0e9e39f249a16976918f6564b8830bc894c89659",
        "required_for_r": false
      },
      "reason": "GatedRepoError; explicitly deferred by user. No placeholder predictions or scores."
    }
  },
  "user_authorization": "User explicitly authorized executing the notebook without Llama, marking it pending until access is granted; complete the available study first.",
  "policy_training_authorized": false,
  "qasper_test_used

Preparing the pinned reader panel: disk check, resumable downloads, then one-reader-at-a-time GPU qualification.


Mistral: complete cache | mistralai/Mistral-7B-Instruct-v0.2 @ 63a8b081895390a26e140280378bc85ec8bce07a
Qwen: complete cache | Qwen/Qwen3-8B @ b968826d9c46dd6066d109eabc6255188de91218


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:  33%|███▎      | 1/3 [00:04<00:08,  4.03s/it]

...

Loading checkpoint shards:  40%|████      | 2/5 [00:01<00:02,  1.38it/s]

Loading checkpoint shards:  60%|██████    | 3/5 [00:02<00:01,  1.16it/s]

Loading checkpoint shards:  80%|████████  | 4/5 [00:03<00:00,  1.15it/s]

Loading checkpoint shards: 100%|██████████| 5/5 [00:03<00:00,  1.28it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Reader qualification passed: Qwen


## Record the reviewed C conclusion before available-reader G

The prior C analysis justifies testing reader utilization on unchanged SFR/evidence. Qualification is required only for the two available readers under the user's explicit deferral; any existing C review is reused. Llama is pending, not removed from the frozen panel.


In [17]:
REVIEW_PHASE = 'C' if READERS_READY else None
REVIEWED_RETRIEVER = 'SFR'
REVIEW_REASON = (
    'Reviewed C: FULL F1=.407095, SARA=.381233, QCS256=.344075, LEAD256=.321490; '
    'FULL-QCS paired CI is positive, while FULL-SARA CI includes zero. '
    'Proceed with the preregistered reader isolation using unchanged SFR/evidence; '
    'the available Mistral/Qwen readers passed qualification; Llama is explicitly pending gated access. The full three-reader study remains incomplete. '
    'No claim of a sole bottleneck, SeleCom reproduction, or confirmed improvement.'
)
if not SMOKE_ONLY: review_if_requested('C')


Development reports saved. Manual evidence-based phase review required.


## W10-G — setup, then resume the frozen reader panel

In [18]:
RUN_READER_FULL = not SMOKE_ONLY and READERS_READY and phase_approved('C')


In [19]:
if RUN_READER_FULL:
    for reader in ACTIVE_READERS:
        run_pending('G-'+reader, 'G', 'G', '--reader', reader)
else: print('PAUSED G: preview mode or reviewed Phase C freeze required.')
print('Llama: PENDING_ACCESS; no predictions or scores fabricated.')


RESUME G-Mistral: 0/693 saved; only pending work will execute.


Validating frozen source/config/data/model identity...


Validating frozen source/config/data/model identity...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:  33%|███▎      | 1/3 [00:00<00:01,  1.28it/s]

...

G/Qwen: 660/693 pending answers saved


G/Qwen: 670/693 pending answers saved


G/Qwen: 680/693 pending answers saved


G/Qwen: 690/693 pending answers saved


Llama: PENDING_ACCESS; no predictions or scores fabricated.


## Phase G reports — automatically refresh after completion

In [20]:
RUN_REPORT = not SMOKE_ONLY


In [21]:
if RUN_REPORT: refresh_reports()


Development reports saved. Manual evidence-based phase review required.


## Record partial G completion before D

After Mistral/Qwen finish, a separate partial-panel receipt permits the prespecified Mistral-only D diagnostic under the user's explicit exception. `freeze_G.json` is not created or replaced: the full three-reader panel remains incomplete. This exception does not authorize policy learning, model selection or TEST evaluation.


In [22]:
G_SURFACE_COMPLETE = all(
    checkpoint_progress(RESULT_ROOT, 'G-'+reader, expected_keys('G'))[1]
    for reader in ACTIVE_READERS
)
PARTIAL_G_READY = not SMOKE_ONLY and G_SURFACE_COMPLETE
if PARTIAL_G_READY:
    partial_stage('complete-G')
    print('Available G panel complete. Full G remains pending Llama; no full-panel freeze is written.')
else:
    print('Available G panel still pending; D stays paused.')


{
  "execution_freeze_hash": "6d603f7220cf6545e64a32227fad18497ea7d56dec8013925d08ab503fbd0488",
  "full_panel_complete": false,
  "available_panel_complete": true,
  "active_readers": [
    "Mistral",
    "Qwen"
  ],
  "coverage": {
    "Mistral": {
      "saved": 693,
      "expected": 693,
      "complete": true
    },
    "Qwen": {
      "saved": 693,
      "expected": 693,
      "complete": true
    }
  },
  "pending_readers": {
    "Llama": {
      "status": "PENDING_ACCESS",
      "model": {
        "id": "meta-llama/Llama-3.1-8B-Instruct",
        "revision": "0e9e39f249a16976918f6564b8830bc894c89659",
        "required_for_r": false
      },
      "reason": "GatedRepoError; explicitly deferred by user. No placeholder predictions or scores."
    }
  },
  "user_authorization": "User explicitly authorized executing the notebook without Llama, marking it pending until access is granted; complete the available study first.",
  "policy_training_authorized": false,
  "qasper_test_use

## W10-D — setup, then resume paired-seed diagnostics

In [23]:
RUN_DATASET_ANALYSIS_FULL = not SMOKE_ONLY and (phase_approved('G') or PARTIAL_G_READY)
RUN_POLICY_EXPERIMENT = False


In [24]:
if RUN_DATASET_ANALYSIS_FULL:
    if PARTIAL_G_READY: run_partial_d()
    else: run_pending('D-stability', 'D', 'D')
else: print('PAUSED D: available reader results required.')
print('Policy training remains disabled. Full reader study remains pending Llama.')


RESUME D under explicit partial-panel authorization: 0/144 saved.


Validating frozen source/config/data/model identity...


Validating frozen source/config/data/model identity...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:  33%|███▎      | 1/3 [00:00<00:01,  1.25it/s]

Loading checkpoint shards:  67%|██████▋   | 2/3 [00:01<00:00,  1.18it/s]

Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.22it/s]


Policy training remains disabled. Full reader study remains pending Llama.


## Final reports and status — always reached unless a real execution/identity error occurs

In [25]:
RUN_REPORT = not SMOKE_ONLY


In [26]:
if RUN_REPORT: refresh_reports()
stage('status')
for phase in ('R', 'C', 'G'):
    print(f'Phase {phase}:', 'review frozen' if phase_approved(phase) else 'awaiting review')
print('Partial study: Mistral/Qwen + D; Llama remains PENDING_ACCESS. Full panel is not complete.')


Development reports saved. Manual evidence-based phase review required.


{
  "C-Mistral": {
    "saved": 1155,
    "complete": true
  },
  "D-stability": {
    "saved": 144,
    "complete": true
  },
  "G-Mistral": {
    "saved": 693,
    "complete": true
  },
  "G-Qwen": {
    "saved": 693,
    "complete": true
  },
  "R-BM25-full": {
    "saved": 924,
    "complete": true
  },
  "R-BM25-sara": {
    "saved": 924,
    "complete": true
  },
  "R-MiniLM-full": {
    "saved": 924,
    "complete": true
  },
  "R-MiniLM-sara": {
    "saved": 924,
    "complete": true
  },
  "R-SFR-full": {
    "saved": 924,
    "complete": true
  },
  "R-SFR-sara": {
    "saved": 924,
    "complete": true
  },
  "R0-BM25": {
    "saved": 231,
    "complete": true
  },
  "R0-MiniLM": {
    "saved": 231,
    "complete": true
  },
  "R0-SFR": {
    "saved": 231,
    "complete": true
  }
}
Phase R: review frozen
Phase C: review frozen
Phase G: awaiting review
Partial study: Mistral/Qwen + D; Llama remains PENDING_ACCESS. Full panel is not complete.


## Completed three-reader study — current results

All three preregistered readers are now complete. Earlier pending-Llama messages above describe the historical two-reader execution; the final report and full-panel receipt supersede that status. Completed answers were reused.

| Reader | No evidence F1 | FULL F1 | QCS256 F1 |
|---|---:|---:|---:|
| Mistral | 0.066586 | 0.235322 | 0.189431 |
| Qwen | 0.073060 | 0.273760 | 0.211520 |
| Llama | 0.068372 | 0.289647 | 0.197178 |

Validated: 231 DEV questions, 86 papers, 8,922 experiment answer rows across R/C/G/D. No new TEST execution or policy training.

[Final Week 10 report](/home/gunavenkat/Downloads/CS787-RAG-Project/results/week10/root_cause/v1/W10_FINAL_REPORT.md) · [Full-panel receipt](/home/gunavenkat/Downloads/CS787-RAG-Project/results/week10/root_cause/v1/full_study_summary.json)
